In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from collections import defaultdict
from tqdm import tqdm

from sklearn.metrics import silhouette_score

from time_series.data_generators import LorenzGenerator
from time_series.models import KernelRidgeRegression, RascuttiModel
from time_series.evaluators import MeanSquaredError

from experiment_logging import Experiment
from time_series_clustering import TimeSeriesClustering

from time_series.data_handlers import TimeSeriesData
import optuna

2026-01-12 16:58:30.222 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering
/home/james/Repo/PhD Repo/time_series_clustering/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_dataset(
    theta: float,
    n_points: int,
    n_correlated_dimensions: int,
    n_uncorrelated_dimensions: int,
    noise: float = 0.0,
    damping: float = 0.9,
    seed: int | None = None,
):
    """
    Generate a bounded nonlinear dynamical system dataset.

    Dynamics are nonlinear but globally stable via tanh damping.
    """
    
    if not (0 < damping < 1):
        raise ValueError("damping must be in (0, 1)")
    if n_points <= 0:
        raise ValueError("n_points must be positive")

    rng = np.random.default_rng(seed)

    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions
    X = np.empty((n_points + 1, n_dim))
    X[0] = rng.uniform(-1.0, 1.0, size=n_dim)

    for t in range(n_points):
        x = X[t]
        x_next = np.zeros_like(x)

        # ------------------------------
        # Correlated nonlinear dynamics
        # ------------------------------
        for i in range(n_correlated_dimensions):
            x_next[i] += np.cos(theta * x[i]) * x[i]

            if i + 1 < n_correlated_dimensions:
                x_next[i] += np.sin(theta * x[i + 1])
                x_next[i + 1] += -np.sin(theta * x[i])

        # ------------------------------
        # Uncorrelated dimensions
        # ------------------------------
        for i in range(n_correlated_dimensions, n_dim):
            x_next[i] = np.cos(theta * x[i]) * x[i]

        # ------------------------------
        # Damping + noise (bounded step)
        # ------------------------------
        x_next = damping * np.tanh(x_next)
        x_next += rng.normal(0.0, noise, size=n_dim)

        X[t + 1] = x_next

    return X


# MSE vs noise

## Kernel Ridge Regression

In [ ]:
n_per = 5
n_groups = 3
seed = 0

n_points=400
n_thetas = 15
n_repeat = 20

In [ ]:
experiment = Experiment(
    "Similarity vs Theta - KRR - TransitionData (Averaged)",
    "experiments"
)

np.random.seed(seed)

experiment.add_config(
    n_points=n_points,
    n_thetas=n_thetas,
    n_repeat=n_repeat,
    seed=seed,
    sweep="theta",
    model="KRR",
    theta_ref=float(np.pi / 4)
)

theta_ref = np.pi / 4
theta_values = np.linspace(0, np.pi / 2, n_thetas)
noise_range = tqdm(np.linspace(1e-3, 10, 20))


for noise in noise_range:
    noise_range.set_description(f"Noise = {noise:.3f}")

    similarities_all = []

    for r in range(n_repeat):
        # ------------------------------
        # Reference dataset
        # ------------------------------
        ref_data = create_dataset(
            theta_ref,
            n_points=n_points,
            n_correlated_dimensions=3,
            n_uncorrelated_dimensions=0,
            noise=noise
        )

        ref_dataset = TimeSeriesData(
            X=ref_data[:-1],
            y=ref_data[1:],
            lag=1,
            train_val_test_split=[0.5, 0.3, 0.2]
        )

        # ------------------------------
        # Theta sweep datasets
        # ------------------------------
        datasets = []
        for theta in theta_values:
            data = create_dataset(
                theta,
                n_points=n_points,
                n_correlated_dimensions=3,
                n_uncorrelated_dimensions=0,
                noise=noise
            )

            datasets.append(
                TimeSeriesData(
                    X=data[:-1],
                    y=data[1:],
                    lag=1,
                    train_val_test_split=[0.5, 0.3, 0.2],
                    theta=theta
                )
            )

        # ------------------------------
        # Hyperparameter optimisation
        # ------------------------------
        def objective(trial):
            bandwidth = trial.suggest_float("bandwidth", 0.1, 4.0)
            reg = trial.suggest_float("reg", 1e-12, 1e-4)

            mse_total = 0.0
            for dataset in datasets:
                X_train, y_train = dataset.train_data()
                X_val, y_val = dataset.val_data()

                model = KernelRidgeRegression(
                    kernel="rbf",
                    bandwidth=bandwidth,
                    reg=reg
                )

                model.fit(X_train, y_train)
                y_pred = model.predict(X_val)
                mse_total += np.mean((y_pred - y_val) ** 2)

            return mse_total

        study = optuna.create_study()
        study.optimize(objective, n_trials=30, n_jobs=-1)
        best_params = study.best_params

        # ------------------------------
        # Fit reference model
        # ------------------------------
        model_ref = KernelRidgeRegression(
            kernel="rbf",
            **best_params
        )

        X_ref, y_ref = ref_dataset.full_data()
        model_ref.fit(X_ref, y_ref)

        # ------------------------------
        # Fit theta models
        # ------------------------------
        similarities = np.zeros(len(theta_values))

        for i, dataset in enumerate(datasets):
            X, y = dataset.full_data()

            model = KernelRidgeRegression(
                kernel="rbf",
                **best_params
            )
            model.fit(X, y)

            similarities[i] = model_ref.inner_product(model)

        similarities_all.append(similarities)

    # ------------------------------
    # Aggregate over repeats
    # ------------------------------
    similarities_all = np.stack(similarities_all)  # (n_repeat, n_thetas)

    mean_similarities = similarities_all.mean(axis=0)
    std_similarities = similarities_all.std(axis=0)

    # ------------------------------
    # Save averaged results
    # ------------------------------
    experiment.add_result(
        **{f"theta_sweep_noise_{noise:.3f}": dict(
            noise=float(noise),
            theta_ref=float(theta_ref),
            theta_values=theta_values.tolist(),
            mean_similarities=mean_similarities.tolist(),
            std_similarities=std_similarities.tolist(),
            n_repeat=n_repeat
        )}
    )


Noise = 0.001:   0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_2446/4236895613.py:145: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  similarities[i] = model_ref.inner_product(model)
Noise = 10.000: 100%|██████████| 20/20 [06:26<00:00, 19.35s/it]


# Dimension Scaling

In [ ]:
theta_ref = np.pi / 4
noise = 1.0

n_points=400
n_thetas = 15
n_repeat = 20
max_correlated_dims = 10

In [ ]:
experiment = Experiment(
    "Similarity vs CorrelatedDims - KRR - TransitionData",
    "experiments"
)

np.random.seed(seed)

experiment.add_config(
    n_points=n_points,
    n_repeat=n_repeat,
    sweep="n_correlated_dimensions",
    model="KRR",
    theta_ref=theta_ref,
    noise=noise
)

# choose your sweep explicitly
correlated_dims_values = np.arange(1, max_correlated_dims + 1)


similarities_all_dims = {}

for n_corr in correlated_dims_values:
    similarities_all = []

    for r in range(n_repeat):
        # ------------------------------
        # Reference dataset
        # ------------------------------
        ref_data = create_dataset(
            theta_ref,
            n_points=n_points,
            n_correlated_dimensions=n_corr,
            n_uncorrelated_dimensions=0,
            noise=noise
        )

        ref_dataset = TimeSeriesData(
            X=ref_data[:-1],
            y=ref_data[1:],
            lag=1,
            train_val_test_split=[0.5, 0.3, 0.2]
        )

        # ------------------------------
        # Comparison dataset (same n_corr)
        # ------------------------------
        data = create_dataset(
            theta_ref,
            n_points=n_points,
            n_correlated_dimensions=n_corr,
            n_uncorrelated_dimensions=0,
            noise=noise
        )

        dataset = TimeSeriesData(
            X=data[:-1],
            y=data[1:],
            lag=1,
            train_val_test_split=[0.5, 0.3, 0.2],
            theta=theta_ref
        )

        # ------------------------------
        # Hyperparameter optimisation
        # ------------------------------
        def objective(trial):
            bandwidth = trial.suggest_float("bandwidth", 0.1, 4.0)
            reg = trial.suggest_float("reg", 1e-12, 1e-4)

            X_train, y_train = dataset.train_data()
            X_val, y_val = dataset.val_data()

            model = KernelRidgeRegression(
                kernel="rbf",
                bandwidth=bandwidth,
                reg=reg
            )

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            return np.mean((y_pred - y_val) ** 2)

        study = optuna.create_study()
        study.optimize(objective, n_trials=30, n_jobs=-1)
        best_params = study.best_params

        # ------------------------------
        # Fit reference model
        # ------------------------------
        model_ref = KernelRidgeRegression(
            kernel="rbf",
            **best_params
        )

        X_ref, y_ref = ref_dataset.full_data()
        model_ref.fit(X_ref, y_ref)

        # ------------------------------
        # Fit comparison model
        # ------------------------------
        model = KernelRidgeRegression(
            kernel="rbf",
            **best_params
        )

        X, y = dataset.full_data()
        model.fit(X, y)

        similarity = model_ref.inner_product(model)
        similarities_all.append(similarity)

    # ------------------------------
    # Aggregate over repeats
    # ------------------------------
    similarities_all = np.array(similarities_all)

    mean_similarity = similarities_all.mean()
    std_similarity = similarities_all.std()

    similarities_all_dims[n_corr] = dict(
        mean=mean_similarity,
        std=std_similarity
    )

    # ------------------------------
    # Save result
    # ------------------------------
    experiment.add_result(
        **{f"corr_dims_{n_corr}": dict(
            n_correlated_dimensions=int(n_corr),
            theta_ref=float(theta_ref),
            noise=noise,
            mean_similarity=float(mean_similarity),
            std_similarity=float(std_similarity),
            n_repeat=n_repeat
        )}
    )


# n uncorrelated dimensions

In [ ]:
theta_ref = np.pi / 4
noise = 1.0
n_correlated = 5

n_points=400
n_thetas = 15
n_repeat = 20
max_uncorrelated_dims = 10

In [ ]:
experiment = Experiment(
    "Similarity vs UncorrelatedDims - KRR - TransitionData",
    "experiments"
)

np.random.seed(seed)

experiment.add_config(
    n_points=n_points,
    n_repeat=n_repeat,
    sweep="n_uncorrelated_dimensions",
    model="KRR",
    theta_ref=float(np.pi / 4),
    noise=1.0,
    n_correlated_dimensions=5
)

# sweep range
uncorrelated_dims_values = np.arange(0, max_uncorrelated_dims + 1)


for n_uncorr in uncorrelated_dims_values:
    similarities_all = []

    for r in range(n_repeat):
        # ------------------------------
        # Reference dataset
        # ------------------------------
        ref_data = create_dataset(
            theta_ref,
            n_points=n_points,
            n_correlated_dimensions=n_correlated,
            n_uncorrelated_dimensions=n_uncorr,
            noise=noise
        )

        ref_dataset = TimeSeriesData(
            X=ref_data[:-1],
            y=ref_data[1:],
            lag=1,
            train_val_test_split=[0.5, 0.3, 0.2]
        )

        # ------------------------------
        # Comparison dataset
        # ------------------------------
        data = create_dataset(
            theta_ref,
            n_points=n_points,
            n_correlated_dimensions=n_correlated,
            n_uncorrelated_dimensions=n_uncorr,
            noise=noise
        )

        dataset = TimeSeriesData(
            X=data[:-1],
            y=data[1:],
            lag=1,
            train_val_test_split=[0.5, 0.3, 0.2],
            theta=theta_ref
        )

        # ------------------------------
        # Hyperparameter optimisation
        # ------------------------------
        def objective(trial):
            bandwidth = trial.suggest_float("bandwidth", 0.1, 4.0)
            reg = trial.suggest_float("reg", 1e-12, 1e-4)

            X_train, y_train = dataset.train_data()
            X_val, y_val = dataset.val_data()

            model = KernelRidgeRegression(
                kernel="rbf",
                bandwidth=bandwidth,
                reg=reg
            )

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            return np.mean((y_pred - y_val) ** 2)

        study = optuna.create_study()
        study.optimize(objective, n_trials=30, n_jobs=-1)
        best_params = study.best_params

        # ------------------------------
        # Fit reference model
        # ------------------------------
        model_ref = KernelRidgeRegression(
            kernel="rbf",
            **best_params
        )

        X_ref, y_ref = ref_dataset.full_data()
        model_ref.fit(X_ref, y_ref)

        # ------------------------------
        # Fit comparison model
        # ------------------------------
        model = KernelRidgeRegression(
            kernel="rbf",
            **best_params
        )

        X, y = dataset.full_data()
        model.fit(X, y)

        similarity = model_ref.inner_product(model)
        similarities_all.append(similarity)

    # ------------------------------
    # Aggregate over repeats
    # ------------------------------
    similarities_all = np.array(similarities_all)

    mean_similarity = similarities_all.mean()
    std_similarity = similarities_all.std()

    # ------------------------------
    # Save result
    # ------------------------------
    experiment.add_result(
        **{f"uncorr_dims_{n_uncorr}": dict(
            n_uncorrelated_dimensions=int(n_uncorr),
            n_correlated_dimensions=n_correlated,
            theta_ref=float(theta_ref),
            noise=noise,
            mean_similarity=float(mean_similarity),
            std_similarity=float(std_similarity),
            n_repeat=n_repeat
        )}
    )
